### Using Pre-existing voice

In [ ]:
import time
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
    dtype=torch.bfloat16,
    device_map="mps",
    attn_implementation="sdpa",
    # device_map="cuda:0",
    # attn_implementation="flash_attention_2",

)

torch.mps.synchronize()
t0 = time.time()

wavs, sr = model.generate_custom_voice(
    text="take a comfortable position and relax",
    language="English", 
    speaker="Ryan",
    instruct="You are reading a guided meditation using your smootest voice", 
)

torch.mps.synchronize()
t1 = time.time()
print(f"[CustomVoice Single] time: {t1 - t0:.3f}s")

sf.write("output_custom_voice.wav", wavs[0], sr)

### Training from reference

In [ ]:
design_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="mps",
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

In [ ]:
# note: 3s clip takes 11.26s to generate
torch.mps.synchronize()
t0 = time.time()

# ref_text = "take a comfortable position and relax"
# ref_instruct = "Male, 45 years old, tenor range. yoga/sleep instructor. talk slowly"

ref_text = "With each step, allow yourself to disengage from whatever has been on your mind. And as you walk, see yourself creating space… a distance from stress, and disruptive thoughts… and transitioning to a state of calm and wellbeing. As you  walk, consider how stepping into a different environment can shift your perspective and allow new ideas to emerge."
ref_instruct = """Female, ~40 years old
Pitch: ~200 Hz (mid–low female range)
Timbre: warm, soft, slightly breathy
Style: calm, controlled, meditative. voice artist for hypnosis and meditation for over 20 years."""
ref_wavs, sr = design_model.generate_voice_design(
    text=ref_text,
    language="English",
    instruct=ref_instruct
)

torch.mps.synchronize()
t1 = time.time()
print(f"[CustomVoice Single] time: {t1 - t0:.3f}s")

sf.write("voice_design_reference.wav", ref_wavs[0], sr)


### Training from cloned voice

https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice

In [ ]:
# took 10s to recreate voice clone
clone_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="mps",
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

torch.mps.synchronize()
t0 = time.time()

voice_clone_prompt = clone_model.create_voice_clone_prompt(
    # ref_audio=(ref_wavs[0], sr),
    ref_audio="/Users/emulie/Data/NarratorVoice/mindful-walk-astere.wav",
    ref_text="Welcome to your mindful walk for wellbeing. Wherever you are and whenever you're ready, go ahead and take your first step, and let's being."
)
torch.save(voice_clone_prompt, "voice_clone_prompt.pt")
# voice_clone_prompt = torch.load("voice_clone_prompt.pt", map_location="mps")  # FIXME


torch.mps.synchronize()
t1 = time.time()
print(f"[Create Voice Clone] time: {t1 - t0:.3f}s")



In [ ]:
# inference time: 81s
sentences = [
    "With each step, allow yourself to disengage from whatever has been on your mind. And as you walk, see yourself creating space… a distance from stress, and disruptive thoughts… and transitioning to a state of calm and wellbeing. As you  walk, consider how stepping into a different environment can shift your perspective and allow new ideas to emerge."
]

torch.mps.synchronize()
t0 = time.time()


# reuse it for multiple single calls
wavs, sr = clone_model.generate_voice_clone(
    text=sentences[0],
    language="English",
    voice_clone_prompt=voice_clone_prompt,
    instruct=" You are a 40 years old female and Head of Wellness at BetterSleep, former journalist, and voice artist, Aster studied hypnosis and meditation for over 20 years. She has written and voiced many of the widely listened to guided hypnosis, and meditations in both English and French. talk slowly"
)

torch.mps.synchronize()
t1 = time.time()
print(f"[CustomVoice Single] time: {t1 - t0:.3f}s")

sf.write("clone_single_1.wav", wavs[0], sr)
